# Swahili news classification with sequential models

Runs the whole project on Google Colab: data, exploratory analysis, five models, comparison and error analysis.

**Before running:** Runtime → Change runtime type → **T4 GPU**, then Runtime → Run all.

Approximate times on a T4: EDA 5 min, baselines 30 min, BiLSTM runs 15 min, TextCNN runs 10 min, AfriBERTa 3 to 4 hours.
Every run appends to `results/experiment_log.csv`. The repository already holds the outputs of our own runs,
so any training cell can be skipped and the comparison cells still work.

In [ ]:
import os

REPO_URL = "https://github.com/irachrist1/swahili-news-classification.git"
REPO_DIR = "swahili-news-classification"

if not os.path.exists("src"):
    if not os.path.exists(REPO_DIR):
        !git clone -q {REPO_URL}
    %cd {REPO_DIR}
!pip install -q -r requirements.txt

In [ ]:
import pandas as pd
from IPython.display import Image, display

pd.set_option("display.max_colwidth", 120)

def show(name, width=800):
    display(Image(filename=f"figures/{name}", width=width))

## 1. Data and exploratory analysis
Downloads the corpus, removes empty, duplicate and train/test-overlapping articles, and makes a stratified validation split.

In [ ]:
!python src/data.py
!python src/eda.py > /dev/null

In [ ]:
for name in ['eda_class_distribution.png', 'eda_lengths.png', 'eda_vocabulary.png', 'eda_positional_information.png', 'eda_code_switching.png']:
    show(name)

## 2. Baselines: TF-IDF with Naive Bayes and Logistic Regression (Dan)

In [ ]:
!python src/baselines.py

## 3. BiLSTM (Rene)
Each run changes one thing from the reference run and logs what it changed and why.

In [ ]:
!python src/train_neural.py --model bilstm --change "Reference run: fastText embeddings fine-tuned, attention pooling, balanced class weights, first 400 words"
!python src/train_neural.py --model bilstm --run-name bilstm_random --embeddings random --change "Random embeddings instead of fastText"
!python src/train_neural.py --model bilstm --run-name bilstm_frozen --freeze-embeddings --change "fastText embeddings frozen"
!python src/train_neural.py --model bilstm --run-name bilstm_maxpool --pooling max --change "Max pooling instead of attention"
!python src/train_neural.py --model bilstm --run-name bilstm_noweights --class-weights none --change "No class weights"

## 4. TextCNN (Thierry)

In [ ]:
!python src/train_neural.py --model textcnn --dropout 0.5 --change "TextCNN (kernels 3,4,5; 100 filters each), fastText embeddings"
!python src/train_neural.py --model textcnn --dropout 0.5 --run-name textcnn_random --embeddings random --change "Random embeddings instead of fastText"
!python src/train_neural.py --model textcnn --dropout 0.5 --run-name textcnn_noweights --class-weights none --change "No class weights"
!python src/train_neural.py --model textcnn --dropout 0.5 --run-name textcnn_wide --num-filters 200 --kernel-sizes 2 3 4 5 --change "200 filters with kernel sizes 2 to 5"
!python src/train_neural.py --model textcnn --dropout 0.5 --run-name textcnn_len200 --max-len 200 --change "Read the first 200 words instead of 400"
!python src/train_neural.py --model textcnn --run-name textcnn_dropout03 --dropout 0.3 --change "Dropout 0.3 instead of 0.5"

## 5. AfriBERTa-small fine-tuning (Gentil)
The longest step. Run `!python src/train_transformer.py --quick` first to check the setup (1 epoch on 2,000 articles).

In [ ]:
!python src/train_transformer.py

## 6. Experiment log

In [ ]:
log = pd.read_csv('results/experiment_log.csv')
log[['run_name', 'model', 'change', 'val_macro_f1', 'test_macro_f1']]

## 7. Model comparison (Dan)
Each model is represented by its best run on validation macro-F1.

In [ ]:
!python src/compare_models.py > /dev/null
pd.read_csv('results/model_comparison.csv')

In [ ]:
pd.read_csv('results/per_class_f1.csv', index_col=0)

In [ ]:
pd.read_csv('results/significance.csv')

In [ ]:
show('experiment_progression.png', 900)
for name in ['confusion_all_models.png', 'per_class_f1.png', 'learning_curves_all.png', 'roc_curves.png', 'calibration.png']:
    show(name, 1000 if name.startswith(('confusion', 'learning')) else 650)

## 8. Error analysis (Thierry)

In [ ]:
!python src/error_analysis.py > /dev/null
show('error_by_length.png')
pd.read_csv('results/error_top_confusions.csv')

In [ ]:
pd.read_csv('results/error_label_audit.csv')

In [ ]:
pd.read_csv('results/error_hard_examples.csv')